# Aiqon Assignment: Train Models

This notebook trains the ETA regression models and the Delay Risk classification models for the shipment dataset. It executes the same code as `scripts/train_models.py` but in an interactive format.

In [1]:
import sys
import os

# Ensure the project root is in the path so we can import from src
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import json

from src import CONFIG, logger
from src.data_loader import load_raw_data
from src.preprocessing import preprocess, save_processed
from src.feature_engineering import (
    build_feature_matrix, NUMERIC_FEATURES, CATEGORICAL_FEATURES,
)
from src.eta_model import train_eta_models, save_model
from src.delay_classifier import train_classifiers, save_classifier

ImportError: DLL load failed while importing _libsvm_sparse: An Application Control policy has blocked this file.

### Phase 1: Load and Preprocess Data

In [ ]:
seed = CONFIG["project"]["random_seed"]
test_size = CONFIG["data"]["test_size"]

print("=== Phase 1: Load & Preprocess ===")
raw = load_raw_data('../data/raw/logistics-shipment-delay-prediction.xlsx')
df = preprocess(raw)
save_processed(df, '../data/processed/shipments_cleaned.csv')

# Filter to records with valid outcomes
df_train_pool = df[df["has_outcome"] == True].copy()
print(f"Records with valid outcome: {len(df_train_pool)}")

# Build feature matrix
X_all, num_cols, cat_cols = build_feature_matrix(df_train_pool)
y_reg = df_train_pool["delay_target_minutes"].values
y_clf = df_train_pool["is_delayed"].astype(int).values

# Train/test split
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = (
    train_test_split(
        X_all, y_reg, y_clf,
        test_size=test_size,
        random_state=seed,
        stratify=y_clf,
    )
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

### Phase 2: Train ETA Regression Models

In [ ]:
print("=== Phase 2: ETA Regression ===")
eta_result = train_eta_models(
    X_train, y_reg_train,
    preprocessor=None,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_name = eta_result["best_name"]
best_model = eta_result["models"][best_name]
feature_cols = num_cols + cat_cols

save_model(
    best_model, best_name,
    eta_result["cv_results"][best_name],
    feature_cols,
    model_dir='../models'
)

print(f"\nETA CV results: {eta_result['cv_results']}")
print(f"Best ETA model: {best_name}")

### Phase 3: Train Delay Classifiers

In [ ]:
print("=== Phase 3: Delay Classification ===")
clf_result = train_classifiers(
    X_train, y_clf_train,
    numeric_cols=num_cols,
    categorical_cols=cat_cols,
    seed=seed,
)

best_clf_name = clf_result["best_name"]
best_clf = clf_result["models"][best_clf_name]

save_classifier(
    best_clf, best_clf_name,
    clf_result["cv_results"][best_clf_name],
    feature_cols,
    model_dir='../models'
)

print(f"\nClassifier CV results: {clf_result['cv_results']}")
print(f"Best classifier: {best_clf_name}")

### Phase 4: Save Split Info for Evaluation

In [ ]:
split_info = {
    "train_indices": X_train.index.tolist(),
    "test_indices": X_test.index.tolist(),
}
split_path = os.path.abspath(os.path.join('..', CONFIG["paths"]["models_dir"], "split_info.json"))
os.makedirs(os.path.dirname(split_path), exist_ok=True)
with open(split_path, "w") as f:
    json.dump(split_info, f)

print("=== Training Complete ===")